In [ ]:
!pip install -U requests pyarrow pandas datasets fastparquet pydantic tqdm groq python-dotenv


# Imports

In [59]:
import itertools
import json
import csv
import asyncio
import re
import os
from dotenv import load_dotenv
from groq import AsyncGroq
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator
from tqdm.auto import tqdm


load_dotenv()
groq_client = AsyncGroq(api_key=os.getenv("GROQ_API_KEY"))


# Configuração

In [60]:
MODELO = "openai/gpt-oss-120b"
CSV_FILE = "../Base-Dataset/dataset_clean.csv"
OUTPUT_DIR = "batches_temp"
FINAL_DATASET = "dataset_finetuning.parquet"
PROGRESS_FILE = "progress.json"

BATCH_SIZE = 50
CONCURRENCY = 1
ITEMS_TO_PROCESS = 1

Path(OUTPUT_DIR).mkdir(exist_ok=True)


In [61]:
class Issue(BaseModel):
    title: str = Field(..., max_length=100)
    body: str = Field(...)
    labels: List[str] = Field(default_factory=lambda: ["technical-debt"])

    @field_validator('body')
    @classmethod
    def clean_newlines_and_validate(cls, v: str) -> str:
        v = v.replace('\\n', '\n').replace('\n', '\n')
        if "###" not in v:
            v = f"### Objetivo\n{v}"
        return v

class IssueList(BaseModel):
    issues: List[Issue]

PROMPT_TEMPLATE = """
Você é um engenheiro de software sênior especialista em arquitetura e gestão de projetos.

Transforme o escopo fornecido em issues do GitHub seguindo o padrão do Angular (ISSUE_TEMPLATE) e Conventional Commits.

### FORMATO OBRIGATÓRIO (JSON VÁLIDO):
{{
  "issues": [
    {{
      "title": "tipo(escopo): mensagem curta",
      "body": "### Objetivo\n...\n\n### Descrição\n...\n\n### Critérios de aceitação\n- [ ] ...\n\n### Tasks\n- [ ] Task 1",
      "labels": ["backend"],
      "keywords": ["k1", "k2", "k3", "k4", "k5"]
    }}
  ]
}}

A resposta é INVÁLIDA se:
- faltar "keywords"
- faltar "labels"
- tiver texto fora do JSON
- tiver múltiplas rotas em uma issue
- não houver diversidade de camadas quando o escopo exigir

---

# REGRAS

## Título
- padrão Conventional Commits: feat, fix, docs, refactor, test, chore
- máximo 100 caracteres
- API → incluir método + rota (ex: POST /users)
- Frontend → incluir nome do componente/tela (ex: DashboardPage)

---

## BODY

### Objetivo
- valor de negócio

### Descrição
- detalhamento técnico REAL

Obrigatório quando aplicável:
- endpoint + payload + response
- tecnologias
- fluxo de execução

Frontend:
- componentes
- estado (hooks, store)
- integração com API

Infra:
- serviços (Docker, Kafka, CI/CD)

Testes:
- tipo (unit, integration, e2e)

---

## TASKS (CRÍTICO)
- NÃO repetir título
- granular (implementável em horas)

Backend:
- DTO
- controller
- service
- persistência

Frontend:
- componente
- estado
- integração API
- loading/erro

---

## LABELS
Escolher entre:
["backend", "frontend", "api", "database", "infra", "ci/cd", "test"]

---

## KEYWORDS (OBRIGATÓRIO)
- mínimo 5
- termos técnicos
- se não gerar → resposta inválida

---

# DIRETRIZES CRÍTICAS

## GRANULARIDADE
- 1 issue = 1 unidade técnica
- 1 rota por issue (PROIBIDO múltiplas)
- 1 componente por issue

---

## FRONTEND (FORÇADO QUANDO APLICÁVEL)
Se o sistema tiver:
- dashboard
- visualização de dados
- interação de usuário
- formulários
- gráficos

ENTÃO:
- gerar issues frontend OBRIGATORIAMENTE

Mapeamento obrigatório:
- GET endpoint → tela de visualização
- POST/PUT → formulário ou ação UI

---

## TESTES (FORÇADO)
Sempre gerar pelo menos:
- 1 test(unit)
- 1 test(e2e OU integration)

---

## INFRA
Separar:
- setup → "infra"
- uso → "backend"

---

## DISTRIBUIÇÃO (ANTI-VIÉS DO MODELO)

Se o escopo envolver sistema completo:

- mínimo esperado:
  - backend issues
  - frontend issues
  - infra issue
  - test issues

Se não houver essa distribuição → resposta inválida

---

## OTIMIZAÇÃO PARA MODELOS LIMITADOS

- Preferir mais issues pequenas do que poucas grandes
- Evitar descrições excessivamente longas
- Priorizar clareza e completude estrutural

---

## REGRAS FINAIS

- Issues independentes
- Mesmo idioma do escopo
- Sem ambiguidade
- APENAS JSON

---

### Escopo:
{escopo}
"""



In [62]:
def formatar_para_treino(input_text, output):
    instruction = "Read the software project description and produce structured GitHub issues..."
    response = json.dumps(output, ensure_ascii=False)
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{response}"


# Gerar Issues

In [63]:
def extrair_json(texto):
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r'(\[.*?\]|\{.*?\})', texto, re.DOTALL)
        if match: 
            try: return json.loads(match.group(1))
            except: pass
    return None

async def gerar_issue_async(texto):
    prompt = PROMPT_TEMPLATE.format(escopo=texto[:2500])
    for tentativa in range(6):
        try:
            response = await groq_client.chat.completions.create(
                model=MODELO,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=4096,
                response_format={"type": "json_object"}
            )
            
            resposta_texto = response.choices[0].message.content
            raw_json = extrair_json(resposta_texto)
            
            if raw_json:
                if isinstance(raw_json, list):
                    data = IssueList(issues=raw_json)
                else:
                    data = IssueList(**raw_json)
                return [i.model_dump() if hasattr(i, 'model_dump') else i.dict() for i in data.issues]
        except Exception as e:
            if tentativa == 5:
                print(f"\n[ERRO CRITICO API] Falha apos multiplas tentativas: {e}")
                raise Exception("FALHA_API")
            await asyncio.sleep(15)
    
    print(f"\n[ERRO DE PARSING] A resposta da IA nao foi validada ou estruturada em JSON corretamente.")
    raise Exception("FALHA_PARSING_JSON")


In [64]:
async def processar_item(texto):
    issues = await gerar_issue_async(texto)
    if not issues:
        return None
    texto_treino = formatar_para_treino(texto, issues)
    print(f"\n────────────────────────────────────────────────\n{texto_treino}\n────────────────────────────────────────────────\n")
    return {"text": texto_treino}


In [65]:
def salvar_batch(batch, index):
    file = f"{OUTPUT_DIR}/dataset_batch_{index}.json"
    with open(file, "w", encoding="utf-8") as f:
        json.dump(batch, f, ensure_ascii=False)
    print(f"[INFO] Batch salvo com sucesso em: {file}")

def consolidar_parquet():
    print("\n[INFO] Iniciando consolidacao final do dataset para formato Parquet...")
    rows = []
    for file in Path(OUTPUT_DIR).glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)
            for item in data:
                if not isinstance(item, dict): continue
                text = item.get("text")
                if not text: continue
                rows.append({"text": str(text)})

    if not rows:
        print("[AVISO] Nenhum exemplo valido encontrado nos batches gerados.")
        return
    
    df = pd.DataFrame(rows)
    df = df.astype({"text": "string"})
    df = df.dropna()
    df.to_parquet(FINAL_DATASET, engine="pyarrow", compression="snappy", index=False)
    print(f"[SUCESSO] Total de {len(df)} exemplos consolidados. Dataset salvo no destino: {FINAL_DATASET}")


In [66]:
async def main():
    textos = []
    print("[INFO] Carregando a base CSV em memoria...")
    with open(CSV_FILE, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("descricao"):
                textos.append(row["descricao"])

    total_textos = len(textos)
    
    start_idx = 0
    batch_index = 0
    if Path(PROGRESS_FILE).exists():
        try:
            with open(PROGRESS_FILE, "r", encoding="utf-8") as pf:
                prog = json.load(pf)
                start_idx = prog.get("last_processed_index", 0)
                batch_index = prog.get("batch_index", 0)
        except:
            pass

    if start_idx >= total_textos:
        print("[INFO] Operacao ignorada: Todos os itens constantes na base ja constam como finalizados.")
        consolidar_parquet()
        return

    end_idx = min(start_idx + ITEMS_TO_PROCESS, total_textos)
    items_this_run = end_idx - start_idx
    print(f"[INFO] Dando continuidade a partir do indice: {start_idx} | Quantidade parametrizada atual: {items_this_run}")

    textos_to_process = textos[start_idx:end_idx]
    batch, tasks = [], []
    pbar = tqdm(total=items_this_run, desc="Progresso Atual")
    
    houve_falha = False
    idx_parada_real = start_idx

    try:
        for i, texto in enumerate(textos_to_process):
            tasks.append(processar_item(texto))
            
            if len(tasks) >= CONCURRENCY or i == len(textos_to_process) - 1:
                resultados = await asyncio.gather(*tasks)
                
                for r in resultados:
                    if r: batch.append(r)
                    if len(batch) >= BATCH_SIZE:
                        salvar_batch(batch, batch_index)
                        batch, batch_index = [], batch_index + 1
                
                idx_parada_real += len(tasks)
                pbar.update(len(tasks))
                tasks = []
                
    except Exception as e:
        houve_falha = True
        print(f"\n[ERRO CRITICO] O loop de execucao interceptou e barrou um padrao de erro fatal da nuvem.")
            
    if batch:
        salvar_batch(batch, batch_index)
        batch_index += 1
        
    pbar.close()

    with open(PROGRESS_FILE, "w", encoding="utf-8") as pf:
        json.dump({"last_processed_index": idx_parada_real, "batch_index": batch_index}, pf)
        
    print(f"[SISTEMA] Estado processual persistido em arquivo log com indice final exato {idx_parada_real}.")

    if not houve_falha and idx_parada_real >= total_textos:
        print("[SISTEMA] Escopo da base total de CSV foi esgotado.")
        consolidar_parquet()


In [67]:
await main()


[INFO] Carregando a base CSV em memoria...
[INFO] Dando continuidade a partir do indice: 2 | Quantidade parametrizada atual: 1


Progresso Atual:   0%|          | 0/1 [00:00<?, ?it/s]


────────────────────────────────────────────────
### Instruction:
Read the software project description and produce structured GitHub issues...

### Input:
OmniChannel is a microservices-based multi-channel marketing automation platform designed to help businesses engage their customers through various communication channels, such as email, SMS, social media, and push notifications. The platform comprises several microservices, including campaign management, audience segmentation, message delivery, and performance analytics. By adopting a microservices architecture, OmniChannel offers a highly customizable and scalable solution that allows businesses to create targeted, personalized marketing campaigns that drive customer engagement and boost sales.

### Response:
[{"title": "feat(campaign): create campaign endpoint", "body": "### Objetivo\nPermitir que usuários criem campanhas de marketing personalizadas via API, aumentando a capacidade de automação.\n\n### Descrição\nImplementar end

# Exportar para CSV

In [68]:
consolidar_parquet() #apenas para testes


[INFO] Iniciando consolidacao final do dataset para formato Parquet...
[SUCESSO] Total de 1 exemplos consolidados. Dataset salvo no destino: dataset_finetuning.parquet


In [69]:
import csv, json
def exportar_para_csv(parquet_path, output_csv):
    print(f"[INFO] Exportando dataset para formato {output_csv}...")
    table = pq.read_table(parquet_path)
    df = table.to_pandas()
    
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(["Title", "Description"])
        for text in df["text"].dropna():
            if "### Response:" not in text: continue
            try:
                json_str = text.split("### Response:")[1].strip()
                data = json.loads(json_str)
                issues = data.get("issues", data) if isinstance(data, dict) else data
                
                for issue in issues:
                    if not isinstance(issue, dict): continue
                    title = issue.get("title", "")
                    body = issue.get("body", "")
                    body = body.replace("\\n", "\n")
                    writer.writerow([title, body])
            except:
                pass
exportar_para_csv(FINAL_DATASET, "gitlab_issues.csv")


[INFO] Exportando dataset para formato gitlab_issues.csv...
